# Completeness & Purity v8 — Plot from JSON
Loads `results_v8.json` and `model_params.json` and re-creates all plots without re-running the pipeline.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy import stats
import os

# ─────────────────────────────────────────────────────────────
# ✏️  EDIT THESE PATHS
RESULTS_JSON   = './v8_results/results_v8.json'
MODEL_JSON     = './v8_results/model_params.json'
OUTPUT_DIR     = './v8_results/plots_rerun'
SAVE_FIGS      = True          # set False to only display
DPI            = 150
# ─────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(RESULTS_JSON) as f:
    R = json.load(f)
with open(MODEL_JSON) as f:
    MP = json.load(f)

pr = R.get('purity', {})
tr = R.get('tradeoff', {})
truth_meta = R.get('truth_completeness', {})
models = MP.get('models', {})

print('Purity keys :', list(pr.keys()))
print('Tradeoff keys:', list(tr.keys()))
print('Model keys   :', list(models.keys()))

In [ ]:
# ─── STYLE HELPERS ───────────────────────────────────────────

# ✏️  Tweak global rcParams here
plt.rcParams.update({
    'font.size': 13,
    'axes.titlesize': 16,
    'axes.labelsize': 15,
    'legend.fontsize': 10,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.facecolor': 'white',
})

COL_RUWE = '#EE6677'
COL_DIST = '#228833'
COL_RV   = '#4477AA'
COL_COMB = '#222222'

def style_ax(ax, minor=True):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis='both', which='major', labelsize=12, width=1.8,
                   length=7, direction='in', top=True, right=True)
    ax.tick_params(axis='both', which='minor', width=1.2,
                   length=4, direction='in', top=True, right=True)
    if minor:
        ax.minorticks_on()
    for item in [ax.xaxis.label, ax.yaxis.label]:
        item.set_fontsize(14)
        item.set_fontweight('bold')
    if ax.get_title():
        ax.title.set_fontsize(15)
        ax.title.set_fontweight('bold')

def savefig(fig, name):
    if SAVE_FIGS:
        path = os.path.join(OUTPUT_DIR, name)
        fig.savefig(path, dpi=DPI, bbox_inches='tight')
        print(f'Saved: {path}')
    plt.show()

# ─── PDF helpers (reconstruct from saved params) ─────────────

def lognorm_core_pdf(x, p, eta_key='eta'):
    eta = p[eta_key]; mu = p['mu_core']; sig = p['sigma_core']
    out = np.zeros_like(x, dtype=float)
    v = x > 0
    out[v] = eta * stats.lognorm.pdf(x[v], s=sig, scale=np.exp(mu))
    return out

def lognorm_tail_pdf(x, p):
    eta = p['eta']; mu = p['mu_tail']; sig = p['sigma_tail']
    out = np.zeros_like(x, dtype=float)
    v = x > 0
    out[v] = (1-eta) * stats.lognorm.pdf(x[v], s=sig, scale=np.exp(mu))
    return out

def gamma_core_pdf(x, p):
    eta = p['eta']; a = p['alpha_core']; b = p['beta_core']
    out = np.zeros_like(x, dtype=float)
    v = x > 0
    out[v] = eta * stats.gamma.pdf(x[v], a=a, scale=1./b)
    return out

def gamma_tail_pdf(x, p):
    eta = p['eta']; a = p['alpha_tail']; b = p['beta_tail']
    out = np.zeros_like(x, dtype=float)
    v = x > 0
    out[v] = (1-eta) * stats.gamma.pdf(x[v], a=a, scale=1./b)
    return out

def gauss_core_pdf(x, p):
    return p['eta'] * stats.norm.pdf(x, p['mu_core'], p['sigma_core'])

def gauss_tail_pdf(x, p):
    return (1-p['eta']) * stats.norm.pdf(x, p['mu_tail'], p['sigma_tail'])

def purity_curve_lognorm(x, p):
    c = lognorm_core_pdf(x, p); t = lognorm_tail_pdf(x, p); tot = c+t
    return np.where(tot > 1e-30, c/tot, 0.5)

def purity_curve_gamma(x, p):
    c = gamma_core_pdf(x, p); t = gamma_tail_pdf(x, p); tot = c+t
    return np.where(tot > 1e-30, c/tot, 0.5)

def purity_curve_gauss(x, p):
    lc = np.log(p['eta']+1e-10) - 0.5*np.log(2*np.pi*p['sigma_core']**2) - 0.5*(x-p['mu_core'])**2/p['sigma_core']**2
    lt = np.log(1-p['eta']+1e-10) - 0.5*np.log(2*np.pi*p['sigma_tail']**2) - 0.5*(x-p['mu_tail'])**2/p['sigma_tail']**2
    lwc = lc; lwt = lt
    return np.exp(lwc - np.logaddexp(lwc, lwt))

print('Helpers ready.')

## Plot 4 — Core Fraction η

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

labels, vals, clrs = [], [], []

if 'ruwe' in pr:
    labels.append('RUWE');      vals.append(pr['ruwe'].get('eta', 0));      clrs.append('#9b59b6')
if 'dist' in pr:
    labels.append('Dist (LN)'); vals.append(pr['dist']['lognormal'].get('eta', 0)); clrs.append('#2196F3')
    labels.append(r'Dist ($\Gamma$)'); vals.append(pr['dist']['gamma'].get('eta', 0)); clrs.append('#F44336')
if 'rv' in pr:
    labels.append('RV (LN)');   vals.append(pr['rv']['lognormal'].get('eta', 0));   clrs.append('#00BCD4')
    labels.append(r'RV ($\Gamma$)');   vals.append(pr['rv']['gamma'].get('eta', 0));   clrs.append('#FF9800')

bars = ax.bar(labels, vals, color=clrs, edgecolor='black', alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.015, f'{v:.3f}',
            ha='center', fontsize=13, fontweight='bold')

ax.set_ylim(0, 1.05)
ax.axhline(0.9, color='red',  ls='--', alpha=0.4, label=r'$\eta$=0.9')
ax.axhline(0.5, color='gray', ls='--', alpha=0.3, label=r'$\eta$=0.5')
ax.set_ylabel(r'Core Fraction $\eta$')
ax.set_title(r'Core Fraction $\eta$ by Model', fontsize=18, fontweight='bold')
ax.legend(fontsize=11)
style_ax(ax)
plt.tight_layout()
savefig(fig, 'plot4_core_fraction_eta.png')

## Plot 5 — Mean Purity by Method

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

labels, vals, clrs = [], [], []

if 'ruwe' in pr:
    labels.append('RUWE');            vals.append(pr['ruwe'].get('mean_purity', 0));    clrs.append('#9b59b6')
if 'dist' in pr:
    labels.append('Dist (LN)');       vals.append(pr['dist'].get('mean_purity_ln', 0)); clrs.append('#2196F3')
    labels.append(r'Dist ($\Gamma$)');vals.append(pr['dist'].get('mean_purity_gm', 0)); clrs.append('#F44336')
if 'rv' in pr:
    labels.append('RV (LN)');         vals.append(pr['rv'].get('mean_purity_ln', 0));   clrs.append('#00BCD4')
    labels.append(r'RV ($\Gamma$)');  vals.append(pr['rv'].get('mean_purity_gm', 0));   clrs.append('#FF9800')
if 'combined' in pr:
    labels.append('Combined');        vals.append(pr['combined'].get('mean', 0));        clrs.append('#222222')

bars = ax.bar(labels, vals, color=clrs, edgecolor='black', alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.015, f'{v:.3f}',
            ha='center', fontsize=13, fontweight='bold')

ax.set_ylim(0, 1.05)
ax.axhline(0.8, color='orange', ls='--', alpha=0.4, label='P=0.8')
ax.set_ylabel('Mean Purity')
ax.set_title('Mean Purity by Method', fontsize=18, fontweight='bold')
ax.legend(fontsize=11)
style_ax(ax)
plt.tight_layout()
savefig(fig, 'plot5_purity_by_method.png')

## Plot 1 — Mixture PDFs (from model params)

In [ ]:
# NOTE: This plot reconstructs PDFs from saved parameters.
# The histograms cannot be reproduced without raw data, so only the model curves are shown.

fig, axes = plt.subplots(2, 3, figsize=(24, 14))
fig.suptitle('Mixture PDFs (reconstructed from model params)', fontsize=20, fontweight='bold', y=0.98)

x_err = np.linspace(0.001, 1.5, 2000)
x_ruwe = np.linspace(0.5, 5, 2000)

# ── Row 0: Log-Normal models ──────────────────────────────────
# RUWE (Gaussian)
ax = axes[0, 0]
if 'ruwe' in models:
    p = models['ruwe']
    ax.fill_between(x_ruwe, gauss_core_pdf(x_ruwe, p), alpha=0.35, color='green',
                    label=f'Core (η={p["eta"]:.2f})')
    ax.fill_between(x_ruwe, gauss_tail_pdf(x_ruwe, p), alpha=0.35, color='red', label='Tail')
    ax.plot(x_ruwe, gauss_core_pdf(x_ruwe, p)+gauss_tail_pdf(x_ruwe, p), 'k--', lw=2, label='Total')
    ax.axvline(1.4, color='orange', ls=':', lw=2, label='RUWE=1.4')
ax.set_xlabel('RUWE'); ax.set_ylabel('Density')
ax.set_title('RUWE — Reg. Gaussian'); ax.set_xlim(0.5, 5)
ax.legend(loc='upper right'); style_ax(ax)

# Dist LN
ax = axes[0, 1]
if 'dist_ln' in models:
    p = models['dist_ln']
    cy = lognorm_core_pdf(x_err, p); ty = lognorm_tail_pdf(x_err, p)
    ax.fill_between(x_err, cy, alpha=0.35, color='green', label=f'Core (η={p["eta"]:.2f})')
    ax.fill_between(x_err, ty, alpha=0.35, color='red', label='Tail')
    ax.plot(x_err, cy+ty, 'k--', lw=2, label='Total')
    cross = np.where((cy[1:] < ty[1:]) & (cy[:-1] >= ty[:-1]))[0]
    if len(cross): ax.axvline(x_err[cross[0]], color='purple', ls='-.', lw=2, label=f'Cross={x_err[cross[0]]:.3f}')
ax.set_xlabel(r'$\sigma_d/d$'); ax.set_ylabel('Density')
ax.set_title(r'$\sigma_d/d$ — Log-Normal'); ax.set_xlim(0, 1.5)
ax.legend(loc='upper right'); style_ax(ax)

# RV LN
ax = axes[0, 2]
if 'rv_ln' in models:
    p = models['rv_ln']
    cy = lognorm_core_pdf(x_err, p); ty = lognorm_tail_pdf(x_err, p)
    ax.fill_between(x_err, cy, alpha=0.35, color='green', label=f'Core (η={p["eta"]:.2f})')
    ax.fill_between(x_err, ty, alpha=0.35, color='red', label='Tail')
    ax.plot(x_err, cy+ty, 'k--', lw=2, label='Total')
ax.set_xlabel(r'$\sigma_v/v$'); ax.set_ylabel('Density')
ax.set_title(r'$\sigma_v/v$ — Log-Normal'); ax.set_xlim(0, 1.5)
ax.legend(loc='upper right'); style_ax(ax)

# ── Row 1: Gamma models ───────────────────────────────────────
ax = axes[1, 0]  # same RUWE Gaussian (bottom row)
if 'ruwe' in models:
    p = models['ruwe']
    ax.fill_between(x_ruwe, gauss_core_pdf(x_ruwe, p), alpha=0.35, color='green', label=f'Core (η={p["eta"]:.2f})')
    ax.fill_between(x_ruwe, gauss_tail_pdf(x_ruwe, p), alpha=0.35, color='red', label='Tail')
    ax.plot(x_ruwe, gauss_core_pdf(x_ruwe, p)+gauss_tail_pdf(x_ruwe, p), 'k--', lw=2, label='Total')
    ax.axvline(1.4, color='orange', ls=':', lw=2, label='RUWE=1.4')
ax.set_xlabel('RUWE'); ax.set_ylabel('Density')
ax.set_title('RUWE — Reg. Gaussian (repeat)'); ax.set_xlim(0.5, 5)
ax.legend(loc='upper right'); style_ax(ax)

# Dist Gamma
ax = axes[1, 1]
if 'dist_gm' in models:
    p = models['dist_gm']
    cy = gamma_core_pdf(x_err, p); ty = gamma_tail_pdf(x_err, p)
    ax.fill_between(x_err, cy, alpha=0.35, color='green', label=f'Core (η={p["eta"]:.2f})')
    ax.fill_between(x_err, ty, alpha=0.35, color='red', label='Tail')
    ax.plot(x_err, cy+ty, 'k--', lw=2, label='Total')
ax.set_xlabel(r'$\sigma_d/d$'); ax.set_ylabel('Density')
ax.set_title(r'$\sigma_d/d$ — Gamma'); ax.set_xlim(0, 1.5)
ax.legend(loc='upper right'); style_ax(ax)

# RV Gamma
ax = axes[1, 2]
if 'rv_gm' in models:
    p = models['rv_gm']
    cy = gamma_core_pdf(x_err, p); ty = gamma_tail_pdf(x_err, p)
    ax.fill_between(x_err, cy, alpha=0.35, color='green', label=f'Core (η={p["eta"]:.2f})')
    ax.fill_between(x_err, ty, alpha=0.35, color='red', label='Tail')
    ax.plot(x_err, cy+ty, 'k--', lw=2, label='Total')
ax.set_xlabel(r'$\sigma_v/v$'); ax.set_ylabel('Density')
ax.set_title(r'$\sigma_v/v$ — Gamma'); ax.set_xlim(0, 1.5)
ax.legend(loc='upper right'); style_ax(ax)

plt.tight_layout(rect=[0, 0, 1, 0.95])
savefig(fig, 'plot1_mixture_pdfs.png')

## Plot 2 — Purity Curves (from model params)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

x_ruwe = np.linspace(0.5, 5, 1000)
x_err  = np.linspace(0.001, 1.5, 1000)

if 'ruwe' in models:
    ax.plot(x_ruwe, purity_curve_gauss(x_ruwe, models['ruwe']),
            color=COL_RUWE, lw=2.5, label='RUWE (RegGauss)')

if 'dist_ln' in models:
    ax.plot(x_err, purity_curve_lognorm(x_err, models['dist_ln']),
            color=COL_DIST, ls='--', lw=2.5, label=r'Dist (LN)')
if 'dist_gm' in models:
    ax.plot(x_err, purity_curve_gamma(x_err, models['dist_gm']),
            color=COL_DIST, ls='-',  lw=2.5, label=r'Dist ($\Gamma$)')

if 'rv_ln' in models:
    ax.plot(x_err, purity_curve_lognorm(x_err, models['rv_ln']),
            color=COL_RV, ls='--', lw=2.5, label=r'RV (LN)')
if 'rv_gm' in models:
    ax.plot(x_err, purity_curve_gamma(x_err, models['rv_gm']),
            color=COL_RV, ls='-',  lw=2.5, label=r'RV ($\Gamma$)')

ax.axhline(0.5, color='gray', ls='--', alpha=0.4)
ax.set_xlabel('Error metric value')
ax.set_ylabel(r'$P_{\rm core}$ (Purity)')
ax.set_title('Purity Curves — All Metrics')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
style_ax(ax)
plt.tight_layout()
savefig(fig, 'plot2_purity_curves.png')

## Plot 3 — Radial Profiles per Quality Cut

In [ ]:
for key, title, xlabel in [
    ('dist', r'$\sigma_d/d$', 'Distance Error'),
    ('rv',   r'$\sigma_v/v$', 'RV Error'),
    ('ruwe', 'RUWE',          'RUWE'),
]:
    rk = key + '_radial'
    gk = key + '_global'
    if rk not in tr or gk not in tr:
        print(f'Skipping {key} (no tradeoff data)')
        continue

    fig, (ax_pur, ax_ret) = plt.subplots(2, 1, figsize=(16, 14), sharex=True)
    rd = tr[rk]; gd = tr[gk]
    rc  = np.array(rd['r_centers'])
    bc  = rd['by_cut']
    cut_keys = sorted(bc.keys(), key=float)
    clrs = plt.cm.viridis(np.linspace(0, 0.9, len(cut_keys)))

    # TOP: purity profile
    for ck, clr in zip(cut_keys, clrs):
        pp = np.array(bc[ck]['purity'])
        vm = np.isfinite(pp)
        if np.sum(vm) > 3:
            ax_pur.plot(rc[vm], pp[vm]*100, '-', color=clr, lw=1.5, label=f'{title}<{float(ck)}')
    ax_pur.set_ylabel('Mean Purity [%]')
    ax_pur.set_title(f'Radial Purity Profile — {xlabel} Cuts', fontsize=18, fontweight='bold')
    ax_pur.set_xscale('log'); ax_pur.set_ylim(0, 105); ax_pur.grid(True, alpha=0.2)
    ax_pur.legend(loc='lower left', fontsize=7, ncol=3)
    style_ax(ax_pur)

    # Inset: global trade-off
    ax_i1 = inset_axes(ax_pur, width='38%', height='35%', loc='upper right', borderpad=2.5)
    ax_i1.patch.set_facecolor('white'); ax_i1.patch.set_alpha(1.0)
    ax_i1.plot(gd['cuts'], [r*100 for r in gd['retention']],       'o-', color='#2196F3', lw=2, ms=4, label='Retention')
    ax_i1.plot(gd['cuts'], [p*100 if not np.isnan(p) else 0 for p in gd['mean_purity']], 's--', color='#F44336', lw=2, ms=4, label='Purity')
    ax_i1.plot(gd['cuts'], [c*100 for c in gd['model_completeness']], 'D:',  color='#228833', lw=2, ms=4, label='Model Compl.')
    ax_i1.set_xlabel(f'{title} cut', fontsize=9, fontweight='bold')
    ax_i1.set_ylabel('%', fontsize=9, fontweight='bold')
    ax_i1.set_title('Trade-off', fontsize=10, fontweight='bold')
    ax_i1.set_ylim(0, 105); ax_i1.legend(fontsize=7, loc='center right'); ax_i1.grid(True, alpha=0.3)
    style_ax(ax_i1)

    # BOTTOM: retention profile
    for ck, clr in zip(cut_keys, clrs):
        rr = np.array(bc[ck]['retention'])
        vm = np.isfinite(rr)
        if np.sum(vm) > 3:
            ax_ret.plot(rc[vm], rr[vm]*100, '--', color=clr, lw=1.5, label=f'{title}<{float(ck)}')
    ax_ret.set_xlabel('Galactocentric Distance R [kpc]')
    ax_ret.set_ylabel('Retention [%]')
    ax_ret.set_title(f'Radial Retention Profile — {xlabel} Cuts', fontsize=18, fontweight='bold')
    ax_ret.set_xscale('log'); ax_ret.set_ylim(0, 105); ax_ret.grid(True, alpha=0.2)
    ax_ret.legend(loc='lower left', fontsize=7, ncol=3)
    style_ax(ax_ret)

    plt.tight_layout()
    savefig(fig, f'plot3_radial_{key}.png')

## Plot 6 — GC/OC/SGR Truth Completeness (Eq. 22)

In [ ]:
truth_colors  = {'GC': '#2ecc71', 'OC': '#3498db', 'SGR': '#e74c3c'}
truth_markers = {'GC': 'o',       'OC': 's',       'SGR': 'D'}

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('GC/OC/SGR Completeness vs Quality Cut (Eq. 22)', fontsize=20, fontweight='bold', y=1.0)

has_truth = any(
    f'truth_completeness_{lab}' in tr.get(gk, {})
    for gk in ['dist_global', 'rv_global', 'ruwe_global']
    for lab in ['GC', 'OC', 'SGR']
)

if not has_truth:
    print('No truth completeness data found in JSON. Skipping Plot 6.')
else:
    for axi, (gk, title, xlabel) in enumerate([
        ('dist_global',  r'$\sigma_d/d$', r'$\sigma_d/d$ cut'),
        ('rv_global',    r'$\sigma_v/v$', r'$\sigma_v/v$ cut'),
        ('ruwe_global',  'RUWE',          'RUWE cut'),
    ]):
        ax = axes[axi]
        if gk not in tr:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            style_ax(ax); continue
        d = tr[gk]
        for lab in ['GC', 'OC', 'SGR']:
            tk = f'truth_completeness_{lab}'
            if tk in d:
                n_tot = truth_meta.get(lab, {}).get('n_total', '?')
                ax.plot(d['cuts'], [c*100 for c in d[tk]],
                        f'{truth_markers[lab]}-',
                        color=truth_colors[lab], lw=2.5, ms=6,
                        label=f'{lab} (N={n_tot:,})' if isinstance(n_tot, int) else f'{lab}')
        ax.plot(d['cuts'], [r*100 for r in d['retention']],        'k--', lw=1.5, alpha=0.5, label='Retention')
        ax.plot(d['cuts'], [c*100 for c in d['model_completeness']],'k:',  lw=1.5, alpha=0.5, label='Model Compl.')
        ax.set_xlabel(xlabel); ax.set_ylabel('Completeness [%]')
        ax.set_title(f'{title} Cuts', fontsize=16, fontweight='bold')
        ax.set_ylim(0, 105); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
        style_ax(ax)

    plt.tight_layout()
    savefig(fig, 'plot6_gc_oc_sgr_completeness.png')

## Bonus: Model Parameter Summary Table

In [ ]:
import pandas as pd

rows = []
for name, p in models.items():
    row = {'model': name}
    row.update({k: round(v, 4) if isinstance(v, float) else v for k, v in p.items()})
    rows.append(row)

df_params = pd.DataFrame(rows).set_index('model')
display(df_params)